# OPRB — Tesseract 5 English LSTM · tessdata_fast · CPU

Đây là **mô hình thứ năm**: Tesseract English LSTM với `eng.traineddata` pretrained từ kho chính thức **tessdata_fast**. Bộ năm hiện tại là Florence, EasyOCR, docTR, PaddleOCR và Tesseract. Không train/fine-tune trên OPRB test.

**Kaggle:** chọn **Accelerator = None**, bật **Internet**, rồi **Run All**. Notebook chạy CPU và có thể chạy song song với notebook GPU khác. Không cần dùng GPU cho Tesseract.

| Thành phần | Cấu hình |
|---|---|
| Dữ liệu | Cùng OPRB test, 6.000 ảnh text + 3.000 ảnh nontext |
| Input | Chỉ ảnh che khuất; GT chỉ dùng để chấm điểm |
| Model | English LSTM `tessdata_fast`, revision và SHA-256 cố định |
| Engine | Tesseract 5 qua `tesserocr==2.13.0`; wheel Linux có sẵn Tesseract/Leptonica |
| Suy luận | OEM 1 (LSTM-only), PSM 3 (auto layout, không OSD); giữ nguyên pixel ảnh; DPI metadata 300 |
| CPU | Giới hạn OpenMP tối đa 2 thread; không đảm bảo engine dùng đủ 2 thread |
| Trạng thái | Model nạp một lần; tắt học thích nghi/từ điển tài liệu và xóa trạng thái từng trang |
| Điểm | Cùng điểm chính đã lọc nhãn GT và điểm `raw_*` như PaddleOCR |

Notebook tự tạo runtime riêng trong `/tmp`; không cài/gỡ PyTorch, CUDA hoặc thư viện hệ thống. Không cần upload thêm `.py`. Mặc định thử **10 ảnh cùng seed 42**; chạy ổn rồi đổi `MAX_SAMPLES_PER_CATEGORY=None` để đánh giá đủ 9.000 ảnh. Sampling cho bài nộp cần được giảng viên chấp thuận.

Checkpoint ghi sau mỗi ảnh, log `PROGRESS` khoảng mỗi phút; vòng đánh giá dừng sau tối đa 10 giờ để xuất phần kết quả đã có. Saved Version mới: gắn output cũ làm Input và điền `RESUME_FROM`. Không trộn checkpoint khác model, CPU/runtime hoặc cấu hình.

Nguồn: [Tesseract pretrained data](https://tesseract-ocr.github.io/tessdoc/Data-Files.html), [tessdata_fast](https://github.com/tesseract-ocr/tessdata_fast), [PSM 3](https://tesseract-ocr.github.io/tessdoc/ImproveQuality.html), [tesserocr wheel và API](https://pypi.org/project/tesserocr/2.13.0/), [OPRB](https://huggingface.co/datasets/kpurkayastha/OPRB).

## 1. Cài runtime CPU riêng

Bật Internet, chọn Accelerator=None. Cell dùng wheel có sẵn nên không cần apt/sudo hay biên dịch C++. Python riêng nằm trong `/tmp`, không được gói vào ZIP. Chỉ cài một lần mỗi phiên, chạy lại sẽ tái dùng runtime đó.

In [ ]:
import importlib.metadata
import json
from pathlib import Path
import subprocess
import sys
import tempfile
import venv

if not (3, 10) <= sys.version_info[:2] <= (3, 14):
    raise RuntimeError('Notebook này hỗ trợ CPython 3.10–3.14; Kaggle hiện dùng Python 3.13.')
pip_version = tuple(int(part) for part in importlib.metadata.version('pip').split('.')[:2])
if pip_version < (22, 3):
    raise RuntimeError('Cần pip >=22.3 để cài vào runtime riêng bằng --python.')
runtime_name = f'oprb_tesseract_py{sys.version_info.major}{sys.version_info.minor}_v1'
TESSERACT_ENV = Path(tempfile.gettempdir()) / runtime_name
TESSERACT_PYTHON = TESSERACT_ENV / 'bin' / 'python'
if not TESSERACT_PYTHON.is_file():
    venv.EnvBuilder(with_pip=False, system_site_packages=False).create(TESSERACT_ENV)
if 'include-system-site-packages = false' not in (TESSERACT_ENV / 'pyvenv.cfg').read_text().lower():
    raise RuntimeError('Tesseract runtime must not inherit system site-packages.')
PIP = [sys.executable, '-m', 'pip', '--python', str(TESSERACT_PYTHON)]
print('Installing CPU runtime:', TESSERACT_ENV, flush=True)
subprocess.check_call(PIP + [
    'install', '-q', '--no-cache-dir', '--only-binary=tesserocr',
    'tesserocr==2.13.0', 'huggingface_hub>=0.34,<1', 'jiwer==4.0.0',
    'numpy>=1.26,<3', 'pandas>=2,<3', 'Pillow>=10,<13',
    'matplotlib>=3.8,<4', 'tqdm', 'wrapt>=1.17,<3',
])
subprocess.check_call([
    str(TESSERACT_PYTHON), '-I', '-c',
    "import importlib.util,tesserocr; assert importlib.util.find_spec('torch') is None; print(tesserocr.tesseract_version())",
])
print('Runtime OK. Tiếp tục Mục 2; giữ Accelerator=None.', flush=True)

## 2. Cấu hình — thử cùng 10 ảnh trước

Giữ seed 42 và cả hai category để so sánh với các model trước. Đổi `MAX_SAMPLES_PER_CATEGORY=None` để chạy đủ test. Khi chọn toàn bộ, chương trình kiểm tra đủ 6.000/3.000 cặp ảnh/GT.

Saved Version mới không tự giữ `/kaggle/working` của phiên cũ: gắn output trước làm Input rồi đặt `RESUME_FROM` đến thư mục có CSV + JSON checkpoint. Giữ model, CPU/runtime, thread limit, DPI và chính sách điểm khi resume.

In [ ]:
from pathlib import Path
import json

MAX_SAMPLES_PER_CATEGORY = 5  # None để chọn toàn bộ 9.000 ảnh.
RESUME_FROM = None  # Ví dụ Path('/kaggle/input/.../oprb_tesseract_eng_fast_cpu_results')
DATA_ROOT = Path('/kaggle/working/oprb_test')
MODEL_DIR = Path('/kaggle/working/tesseract_fast_models')
OUTPUT_DIR = Path('/kaggle/working/oprb_tesseract_eng_fast_cpu_results')
CATEGORIES = ['text', 'nontext']
RANDOM_SEED = 42
MAX_EVAL_HOURS = 10.0
CPU_THREADS = 2  # Giới hạn OpenMP, có hiệu lực trong process mới mỗi lần chạy Mục 3.
SOURCE_DPI = 300  # Metadata cho engine, không resize ảnh.
OCR_TIMEOUT_MS = 120000  # 120 giây tối đa cho bước Recognize của một trang.
DOWNLOAD_DATA = True  # False khi DATA_ROOT đã chứa img/*.jpg và gt/*.txt của test.

assert CATEGORIES and set(CATEGORIES) <= {'text', 'nontext'} and len(CATEGORIES) == len(set(CATEGORIES))
assert MAX_SAMPLES_PER_CATEGORY is None or MAX_SAMPLES_PER_CATEGORY > 0
assert MAX_EVAL_HOURS is None or MAX_EVAL_HOURS > 0
assert CPU_THREADS > 0 and SOURCE_DPI > 0 and OCR_TIMEOUT_MS > 0
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_SETTINGS = {
    'repo_id': 'kpurkayastha/OPRB', 'data_root': str(DATA_ROOT), 'model_dir': str(MODEL_DIR),
    'output_dir': str(OUTPUT_DIR), 'categories': CATEGORIES,
    'max_samples_per_category': MAX_SAMPLES_PER_CATEGORY, 'random_seed': RANDOM_SEED,
    'normalize_case': True, 'max_eval_hours': MAX_EVAL_HOURS, 'cpu_threads': CPU_THREADS,
    'page_segmentation_mode': 3, 'source_dpi': SOURCE_DPI, 'ocr_timeout_ms': OCR_TIMEOUT_MS,
    'download_data': DOWNLOAD_DATA, 'resume_from': str(RESUME_FROM) if RESUME_FROM is not None else None,
}
print('Device: CPU; OpenMP thread limit:', CPU_THREADS, flush=True)
print('Samples/category:', MAX_SAMPLES_PER_CATEGORY, '; seed:', RANDOM_SEED, flush=True)
print('Output:', OUTPUT_DIR, flush=True)

## 3. Chạy và theo dõi PROGRESS

Notebook chứa sẵn toàn bộ chương trình. Cell đầu lưu `.py`/JSON vào output; cell tiếp theo chạy bằng Python riêng và chuyển log về đây hoặc View Logs của Saved Version. Model được nạp một lần trong process đó.

Thời gian/ảnh gồm đọc ảnh, tiền xử lý nội bộ, nhận dạng, lấy transcript/confidence và xóa trạng thái trang. Không gồm tải model/dataset, khởi tạo API, warm-up, chấm điểm, checkpoint hoặc chuyển log. CPU và GPU khác phần cứng: ghi đúng thiết bị khi so sánh thời gian.

Checkpoint ghi sau từng trang. Giới hạn 10 giờ kết thúc vòng đánh giá để xuất kết quả một phần. Interrupt giữ các hàng đã ghi xong; chạy lại Mục 3 để tiếp tục.

In [ ]:
import hashlib

BENCHMARK_SOURCE = 'from __future__ import annotations\nimport argparse\nimport importlib.metadata\nimport importlib.util\nimport json\nimport os\nfrom pathlib import Path\nimport sys\n\nparser = argparse.ArgumentParser(description=\'OPRB Tesseract English LSTM tessdata_fast benchmark\')\nparser.add_argument(\'--config\', required=True)\nargs = parser.parse_args()\nSETTINGS = json.loads(Path(args.config).read_text(encoding=\'utf-8\'))\nif sys.prefix == sys.base_prefix:\n    raise RuntimeError(\'Run this script using the isolated Tesseract Python environment.\')\nif importlib.util.find_spec(\'torch\') is not None:\n    raise RuntimeError(\'The isolated Tesseract runtime must not inherit system site-packages.\')\nTHREAD_LIMIT = int(SETTINGS.get(\'cpu_threads\', 2))\nif THREAD_LIMIT < 1:\n    raise ValueError(\'cpu_threads must be positive\')\n# Set before importing NumPy/Tesseract and their native OpenMP libraries.\nos.environ[\'OMP_THREAD_LIMIT\'] = str(THREAD_LIMIT)\nos.environ[\'OMP_NUM_THREADS\'] = str(THREAD_LIMIT)\nos.environ[\'MPLBACKEND\'] = \'Agg\'\nos.environ.setdefault(\'MPLCONFIGDIR\', \'/tmp/oprb_tesseract_matplotlib\')\nprint(\'ISOLATED CPU RUNTIME:\', sys.executable, \'; Python\', sys.version.split()[0], flush=True)\n\nprint(\'Import Tesseract and inspect CPU\', flush=True)\n\nimport atexit, gc, hashlib, json, os, platform, random, re, shutil, tarfile, time, unicodedata\nimport urllib.request\nfrom collections import Counter\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport tesserocr\nfrom PIL import Image\nfrom huggingface_hub import hf_hub_download\nfrom jiwer import cer, wer\nfrom tqdm import tqdm\nimport matplotlib.pyplot as plt\n\n\ndef display(value):\n    if isinstance(value, pd.DataFrame):\n        print(value.to_string(index=False), flush=True)\n\n\nif version(\'tesserocr\') != \'2.13.0\':\n    raise RuntimeError(\'Expected the pinned binary wheel tesserocr==2.13.0.\')\nENGINE_VERSION_FULL = tesserocr.tesseract_version()\nif not re.match(r\'tesseract 5\\.\', ENGINE_VERSION_FULL):\n    raise RuntimeError(f\'Expected Tesseract 5; found {ENGINE_VERSION_FULL}\')\ncpu_name = platform.processor().strip()\ncpuinfo = Path(\'/proc/cpuinfo\')\nif cpuinfo.is_file():\n    for line in cpuinfo.read_text(errors=\'replace\').splitlines():\n        if line.lower().startswith(\'model name\'):\n            cpu_name = line.split(\':\', 1)[1].strip()\n            break\nCPU_INFO = {\n    \'name\': cpu_name or \'CPU\',\n    \'machine\': platform.machine(),\n    \'logical_cpu_count_reported\': os.cpu_count(),\n    \'affinity_cpu_count\': len(os.sched_getaffinity(0)) if hasattr(os, \'sched_getaffinity\') else None,\n    \'openmp_thread_limit_requested\': THREAD_LIMIT,\n}\nprint(\'Tesserocr:\', version(\'tesserocr\'), flush=True)\nprint(ENGINE_VERSION_FULL, flush=True)\nprint(\'CPU:\', json.dumps(CPU_INFO), flush=True)\n\n\nprint(\'Configuration\', flush=True)\n\nREPO_ID = SETTINGS.get(\'repo_id\', \'kpurkayastha/OPRB\')\nMODEL_ID = \'tesseract-ocr/tessdata_fast:eng:LSTM\'\nDATA_ROOT = Path(SETTINGS[\'data_root\'])\nMODEL_DIR = Path(SETTINGS[\'model_dir\'])\nOUTPUT_DIR = Path(SETTINGS[\'output_dir\'])\nRESUME_FROM = Path(SETTINGS[\'resume_from\']) if SETTINGS.get(\'resume_from\') else None\nCATEGORIES = SETTINGS[\'categories\']\nMAX_SAMPLES_PER_CATEGORY = SETTINGS[\'max_samples_per_category\']\nRANDOM_SEED = SETTINGS.get(\'random_seed\', 42)\nNORMALIZE_CASE = SETTINGS.get(\'normalize_case\', True)\nMAX_EVAL_HOURS = SETTINGS.get(\'max_eval_hours\', 10.0)\nDEVICE = \'cpu\'\nDEVICE_NAME = CPU_INFO[\'name\']\nCPU_THREADS = THREAD_LIMIT\nGT_LAYOUT_MARKERS = [\'##LTLine##\', \'##LTFigure##\']\nMETRIC_POLICY_ID = \'oprb_macro_nfkc_layout_markers_removed_v1\'\nLANGUAGE = \'eng\'\nPAGE_SEGMENTATION_MODE = int(SETTINGS.get(\'page_segmentation_mode\', 3))\nSOURCE_DPI = int(SETTINGS.get(\'source_dpi\', 300))\nOCR_TIMEOUT_MS = int(SETTINGS.get(\'ocr_timeout_ms\', 120000))\nassert PAGE_SEGMENTATION_MODE == 3, \'This benchmark uses automatic page segmentation without OSD (PSM 3).\'\nassert SOURCE_DPI > 0 and OCR_TIMEOUT_MS > 0\nassert MAX_EVAL_HOURS is None or MAX_EVAL_HOURS > 0\nassert MAX_SAMPLES_PER_CATEGORY is None or MAX_SAMPLES_PER_CATEGORY > 0\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\nMODEL_DIR.mkdir(parents=True, exist_ok=True)\nprint(\'Model:\', MODEL_ID, \'; CPU:\', DEVICE_NAME, flush=True)\nprint(\'Samples/category:\', MAX_SAMPLES_PER_CATEGORY, \'; seed:\', RANDOM_SEED, flush=True)\nprint(\'Output:\', OUTPUT_DIR, flush=True)\n\nruntime_packages = {\n    distribution.metadata[\'Name\']: distribution.version\n    for distribution in importlib.metadata.distributions()\n    if distribution.metadata.get(\'Name\')\n}\n(OUTPUT_DIR / \'runtime_packages.json\').write_text(json.dumps(dict(sorted(runtime_packages.items())), indent=2), encoding=\'utf-8\')\n\n\nprint(\'Download test images and scoring GT\', flush=True)\ndef extract_flat(archive_path: str, destination: Path, allowed_suffix: str) -> int:\n    """Safely copy only files of the specified type; never use archive-supplied paths."""\n    destination.mkdir(parents=True, exist_ok=True)\n    count = 0\n    with tarfile.open(archive_path, mode=\'r:gz\') as archive:\n        for member in tqdm(archive, desc=f\'Extract {destination.parent.name}/{destination.name}\', leave=False):\n            if not member.isfile() or not member.name.lower().endswith(allowed_suffix):\n                continue\n            filename = Path(member.name).name\n            target = destination / filename\n            if not target.exists():\n                stream = archive.extractfile(member)\n                if stream is None:\n                    continue\n                with stream, target.open(\'wb\') as out:\n                    shutil.copyfileobj(stream, out)\n            count += 1\n    return count\n\ndef download_test_category(category: str):\n    if category not in {\'text\', \'nontext\'}:\n        raise ValueError(f\'Unknown category: {category}\')\n    for kind, suffix in ((\'img\', \'.jpg\'), (\'gt\', \'.txt\')):\n        repo_path = f\'test/{category}/{kind}.tar.gz\'\n        target_dir = DATA_ROOT / category / kind\n        existing = list(target_dir.glob(f\'*{suffix}\')) if target_dir.exists() else []\n        expected = 6000 if category == \'text\' else 3000\n        if len(existing) >= expected:\n            print(f\'Reuse {category}/{kind}: {len(existing)} files\')\n            continue\n        print(f\'Downloading {repo_path} ...\')\n        tar_path = hf_hub_download(repo_id=REPO_ID, repo_type=\'dataset\', filename=repo_path)\n        n = extract_flat(tar_path, target_dir, suffix)\n        print(f\'  {category}/{kind}: {n:,} archive members of type {suffix} -> {target_dir}\')\nif SETTINGS.get(\'download_data\', True):\n    for category in CATEGORIES:\n        download_test_category(category)\nelse:\n    print(\'Using local test images/GT from:\', DATA_ROOT, flush=True)\n\nprint(\'Pair test images and GT\', flush=True)\ndef parse_occlusion_metadata(stem: str):\n    # OPRB filename convention: OcclusionType__density__batchID... \n    parts = stem.split("__", 2)\n    return {\n        "occlusion_type": parts[0] if parts else "unknown",\n        "occlusion_density": parts[1] if len(parts) > 1 else "unknown",\n    }\n\n\ndef collect_test_pairs(categories):\n    pairs = []\n    for category in categories:\n        image_dir = DATA_ROOT / category / "img"\n        gt_dir = DATA_ROOT / category / "gt"\n        images = {p.stem: p for p in image_dir.glob("*.jpg")}\n        gts = {p.stem: p for p in gt_dir.glob("*.txt")}\n        common = sorted(images.keys() & gts.keys())\n        print(f"{category}: images={len(images):,}, gt={len(gts):,}, paired={len(common):,}")\n        if not common:\n            raise RuntimeError(f"No paired files for {category}. Check extraction and naming.")\n        if MAX_SAMPLES_PER_CATEGORY is not None:\n            rng = random.Random(RANDOM_SEED)\n            common = sorted(rng.sample(common, min(MAX_SAMPLES_PER_CATEGORY, len(common))))\n        for stem in common:\n            pairs.append({\n                "category": category,\n                "sample_id": stem,\n                "image_path": str(images[stem]),\n                "gt_path": str(gts[stem]),\n                **parse_occlusion_metadata(stem),\n            })\n    return pairs\n\n\npairs = collect_test_pairs(CATEGORIES)\nprint(f"Selected {len(pairs):,} samples. Set MAX_SAMPLES_PER_CATEGORY=None to use the entire test set.")\ndisplay(pd.DataFrame(pairs)[["category", "occlusion_type", "occlusion_density", "sample_id"]].head(8))\n\nwith Image.open(pairs[0]["image_path"]) as image:\n    preview = image.copy()\npreview.thumbnail((650, 850))\ndisplay(preview)\nif MAX_SAMPLES_PER_CATEGORY is None:\n    expected_counts = {\'text\': 6000, \'nontext\': 3000}\n    actual_counts = Counter(sample[\'category\'] for sample in pairs)\n    for category in CATEGORIES:\n        if actual_counts[category] != expected_counts[category]:\n            raise RuntimeError(f\'Full test requires {expected_counts[category]} paired {category} pages; got {actual_counts[category]}.\')\npreview.save(OUTPUT_DIR / \'preview.png\')\n\nprint(\'Metrics: layout tokens removed + legacy raw scores\', flush=True)\ndef read_gt_tokens(gt_path: str):\n    tokens = []\n    with open(gt_path, \'r\', encoding=\'utf-8-sig\', errors=\'replace\') as file:\n        for line in file:\n            fields = line.rstrip(\'\\r\\n\').split(\'\\t\')\n            if len(fields) >= 10 and fields[0].strip():\n                tokens.append(fields[0].strip())\n    if not tokens:\n        raise ValueError(f\'No GT tokens read from {gt_path}. Check annotation format.\')\n    return tokens\n\n\ndef normalize_text(text: str) -> str:\n    text = unicodedata.normalize(\'NFKC\', text)\n    text = re.sub(r\'\\s+\', \' \', text).strip()\n    return text.casefold() if NORMALIZE_CASE else text\n\n\ndef clean_text(text: str) -> str:\n    normalized = normalize_text(text)\n    markers = {normalize_text(marker) for marker in GT_LAYOUT_MARKERS}\n    return \' \'.join(token for token in normalized.split() if token not in markers)\n\n\ndef bag_of_words_prf(reference: str, prediction: str):\n    ref, hyp = Counter(reference.split()), Counter(prediction.split())\n    matched = sum((ref & hyp).values())\n    ref_n, hyp_n = sum(ref.values()), sum(hyp.values())\n    precision = matched / hyp_n if hyp_n else (1.0 if ref_n == 0 else 0.0)\n    recall = matched / ref_n if ref_n else (1.0 if hyp_n == 0 else 0.0)\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n    return precision, recall, f1\n\n\ndef score_normalized_pair(ref: str, hyp: str):\n    if not ref:\n        raise ValueError(\'Empty GT transcript after normalization; cannot compute CER/WER.\')\n    precision, recall, f1 = bag_of_words_prf(ref, hyp)\n    return {\n        \'cer\': float(cer(ref, hyp)), \'wer\': float(wer(ref, hyp)),\n        \'token_precision\': precision, \'token_recall\': recall, \'token_f1\': f1,\n        \'gt_words\': len(ref.split()), \'pred_words\': len(hyp.split()),\n    }\n\n\ndef compute_metrics(ground_truth: str, prediction: str):\n    ref_raw, hyp_raw = normalize_text(ground_truth), normalize_text(prediction)\n    ref, hyp = clean_text(ground_truth), clean_text(prediction)\n    return {\n        **score_normalized_pair(ref, hyp),\n        **{f\'raw_{key}\': value for key, value in score_normalized_pair(ref_raw, hyp_raw).items()},\n        \'n_gt_markers_removed\': len(ref_raw.split()) - len(ref.split()),\n        \'n_pred_markers_removed\': len(hyp_raw.split()) - len(hyp.split()),\n    }\n\n\nassert compute_metrics(\'The cat sat\', \'The cat\')[\'wer\'] == 1/3\nassert compute_metrics(\'Hello ##LTLine## world\', \'hello world\')[\'wer\'] == 0\nassert compute_metrics(\'Hello ##LTLine## world\', \'hello world\')[\'raw_wer\'] == 1/3\nprint(\'Toy check:\', compute_metrics(\'Hello ##LTLine## world\', \'hello world\'), flush=True)\nprint(\'First GT tokens:\', read_gt_tokens(pairs[0][\'gt_path\'])[:35], flush=True)\n\nprint(\'Load official English tessdata_fast pretrained model\', flush=True)\n\nMODEL_REVISION = \'87416418657359cb625c412a48b6e1d6d41c29bd\'\nMODEL_URL = \'https://raw.githubusercontent.com/tesseract-ocr/tessdata_fast/87416418657359cb625c412a48b6e1d6d41c29bd/eng.traineddata\'\nMODEL_SHA256 = \'7d4322bd2a7749724879683fc3912cb542f19906c83bcc1a52132556427170b2\'\nMODEL_EXPECTED_BYTES = 4113088\nTRAINEDDATA = MODEL_DIR / \'eng.traineddata\'\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with path.open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\nif not TRAINEDDATA.is_file() or sha256_file(TRAINEDDATA) != MODEL_SHA256:\n    temporary = TRAINEDDATA.with_suffix(\'.traineddata.download\')\n    request = urllib.request.Request(MODEL_URL, headers={\'User-Agent\': \'OPRB-Tesseract-Notebook/1.0\'})\n    print(\'Downloading official eng tessdata_fast...\', flush=True)\n    with urllib.request.urlopen(request, timeout=60) as response, temporary.open(\'wb\') as stream:\n        shutil.copyfileobj(response, stream)\n    if temporary.stat().st_size != MODEL_EXPECTED_BYTES or sha256_file(temporary) != MODEL_SHA256:\n        temporary.unlink(missing_ok=True)\n        raise ValueError(\'Downloaded traineddata has an unexpected size/checksum.\')\n    temporary.replace(TRAINEDDATA)\n\nMODEL_COMPONENTS = [{\n    \'component\': \'english_LSTM_and_language_data\',\n    \'architecture\': \'Tesseract 5 English LSTM (tessdata_fast)\',\n    \'checkpoint_repo\': \'tesseract-ocr/tessdata_fast\',\n    \'checkpoint_revision\': MODEL_REVISION,\n    \'checkpoint_url\': MODEL_URL,\n    \'checkpoint_filename\': TRAINEDDATA.name,\n    \'sha256\': sha256_file(TRAINEDDATA),\n    \'weight_file_bytes\': TRAINEDDATA.stat().st_size,\n    \'parameter_count\': None,\n    \'parameter_count_note\': \'traineddata includes language dictionaries and network; do not infer count from file size.\',\n    \'pretrained_weight_format\': \'Integer LSTM weights from the official tessdata_fast repository\',\n}]\npackage_dir = Path(tesserocr.__file__).resolve().parent\nnative_files = sorted(package_dir.rglob(\'*.so\'))\nbundled_dir = package_dir.parent / \'tesserocr.libs\'\nif bundled_dir.is_dir():\n    native_files += sorted(path for path in bundled_dir.iterdir() if path.is_file() and \'.so\' in path.name)\nNATIVE_ARTIFACTS = [{\'filename\': str(path.relative_to(package_dir.parent)), \'sha256\': sha256_file(path)} for path in native_files]\n\napi = tesserocr.PyTessBaseAPI(\n    path=str(MODEL_DIR), lang=LANGUAGE,\n    oem=tesserocr.OEM.LSTM_ONLY, psm=tesserocr.PSM.AUTO,\n)\natexit.register(api.End)\n# Fixed pretrained inference, with no cross-page classifier/document-dictionary learning.\nfor key in (\'classify_enable_learning\', \'tessedit_enable_doc_dict\'):\n    if not api.SetVariable(key, \'0\'):\n        raise RuntimeError(f\'Cannot disable adaptive option: {key}\')\nif api.GetLoadedLanguages() != [\'eng\'] or int(api.GetPageSegMode()) != PAGE_SEGMENTATION_MODE:\n    raise RuntimeError(\'Unexpected language or page segmentation mode.\')\nEFFECTIVE_OCR_SETTINGS = {\n    \'oem\': int(tesserocr.OEM.LSTM_ONLY),\n    \'psm\': int(api.GetPageSegMode()),\n    \'language\': LANGUAGE,\n    \'source_dpi\': SOURCE_DPI,\n    \'image_resizing\': None,\n    \'recognition_timeout_ms\': OCR_TIMEOUT_MS,\n    \'openmp_thread_limit_requested\': CPU_THREADS,\n    \'classify_enable_learning\': api.GetVariableAsString(\'classify_enable_learning\'),\n    \'tessedit_enable_doc_dict\': api.GetVariableAsString(\'tessedit_enable_doc_dict\'),\n    \'cross_page_state\': \'ClearAdaptiveClassifier before each page; Clear after each page\',\n    \'confidence_filter\': None,\n    \'initialization\': \'One PyTessBaseAPI per run; pretrained weights stay loaded\',\n}\n(OUTPUT_DIR / \'model_manifest.json\').write_text(json.dumps({\n    \'components\': MODEL_COMPONENTS, \'native_runtime_artifacts\': NATIVE_ARTIFACTS,\n    \'engine_version_full\': ENGINE_VERSION_FULL, \'ocr_settings\': EFFECTIVE_OCR_SETTINGS,\n}, indent=2), encoding=\'utf-8\')\nprint(\'Loaded:\', MODEL_COMPONENTS[0][\'architecture\'], flush=True)\nprint(\'Traineddata MiB:\', TRAINEDDATA.stat().st_size / 1024**2, flush=True)\n\n\nprint(\'Single-image warm-up\', flush=True)\n\ndef transcribe_page(image_path: str):\n    """Only an occluded-image path reaches the model; never GT or GT boxes."""\n    started = time.perf_counter()\n    try:\n        api.ClearAdaptiveClassifier()\n        api.SetImageFile(str(image_path))\n        # Source resolution metadata only; image pixels are not resized.\n        api.SetSourceResolution(SOURCE_DPI)\n        if not api.Recognize(timeout=OCR_TIMEOUT_MS):\n            raise RuntimeError(\'Tesseract recognition failed or reached the per-page timeout.\')\n        prediction = api.GetUTF8Text()\n        confidences = api.AllWordConfidences()\n        if not isinstance(prediction, str) or not isinstance(confidences, list):\n            raise TypeError(\'Unexpected Tesseract text/word-confidence output.\')\n        n_regions = len(confidences)\n    finally:\n        api.Clear()\n    elapsed = time.perf_counter() - started\n    return prediction, elapsed, n_regions\n\n\nsample = pairs[0]\nexample_gt = \' \'.join(read_gt_tokens(sample[\'gt_path\']))\nexample_prediction, example_time, example_regions = transcribe_page(sample[\'image_path\'])\nprint(\'CATEGORY:\', sample[\'category\'], flush=True)\nprint(\'IMAGE:\', Path(sample[\'image_path\']).name, flush=True)\nprint(\'LATENCY (s):\', round(example_time, 3), flush=True)\nprint(\'OCR WORDS:\', example_regions, flush=True)\nprint(\'GT RAW (first 700 chars):\', example_gt[:700], flush=True)\nprint(\'PREDICTION (first 700 chars):\', example_prediction[:700], flush=True)\nprint(\'METRICS:\', compute_metrics(example_gt, example_prediction), flush=True)\n\n\nprint(\'Evaluation and checkpoint\', flush=True)\nRESULTS_CSV = OUTPUT_DIR / \'per_image_results.csv\'\nCHECKPOINT_CONFIG = OUTPUT_DIR / \'checkpoint_config.json\'\ncheckpoint_settings = {\n    \'dataset\': REPO_ID, \'split\': \'test\',\n    \'model\': MODEL_ID, \'components\': MODEL_COMPONENTS,\n    \'runtime_isolation\': \'venv_without_system_site_packages\',\n    \'python_version\': platform.python_version(),\n    \'tesserocr_version\': version(\'tesserocr\'),\n    \'tesseract_version_full\': ENGINE_VERSION_FULL,\n    \'native_runtime_artifacts\': NATIVE_ARTIFACTS,\n    \'language\': LANGUAGE, \'device\': DEVICE, \'device_name\': DEVICE_NAME,\n    \'cpu_threads_limit\': CPU_THREADS,\n    \'ocr_settings\': EFFECTIVE_OCR_SETTINGS,\n    \'reading_order\': \'Tesseract GetUTF8Text native order; whitespace normalized only while scoring\',\n    \'metric_policy_id\': METRIC_POLICY_ID,\n    \'excluded_layout_tokens\': GT_LAYOUT_MARKERS,\n    \'normalize_case\': NORMALIZE_CASE,\n    \'metric_definition\': \'Macro per-image metrics after layout-token removal; raw_* retains legacy unfiltered scores\',\n    \'numpy_version\': np.__version__, \'pandas_version\': version(\'pandas\'), \'jiwer_version\': version(\'jiwer\'),\n}\n\nif RESUME_FROM is not None and not RESULTS_CSV.exists():\n    resume_root = Path(RESUME_FROM)\n    for filename in (\'checkpoint_config.json\', \'per_image_results.csv\'):\n        if not (resume_root / filename).is_file():\n            raise FileNotFoundError(f\'Missing resume file: {resume_root / filename}\')\n    for filename in (\'checkpoint_config.json\', \'per_image_results.csv\'):\n        source = resume_root / filename\n        target = OUTPUT_DIR / filename\n        if target.exists() and target.read_bytes() != source.read_bytes():\n            raise FileExistsError(f\'Không ghi đè checkpoint khác: {target}\')\n        if source.resolve() != target.resolve():\n            shutil.copy2(source, target)\n    print(\'Restored checkpoint from:\', resume_root, flush=True)\n\nif CHECKPOINT_CONFIG.exists():\n    saved_settings = json.loads(CHECKPOINT_CONFIG.read_text(encoding=\'utf-8\'))\n    if saved_settings != checkpoint_settings:\n        raise ValueError(\'Checkpoint khác mô hình/cấu hình/thiết bị. Chọn OUTPUT_DIR mới cho thí nghiệm khác.\')\nelif RESULTS_CSV.exists():\n    raise ValueError(\'CSV thiếu checkpoint_config.json. Phục hồi cả hai tệp hoặc chọn OUTPUT_DIR mới.\')\nelse:\n    CHECKPOINT_CONFIG.write_text(json.dumps(checkpoint_settings, indent=2), encoding=\'utf-8\')\n\n\ndef sample_key(record):\n    return record[\'category\'], record[\'sample_id\']\n\n\nselected_keys = {sample_key(sample) for sample in pairs}\nif RESULTS_CSV.exists():\n    previous = pd.read_csv(RESULTS_CSV, keep_default_na=False).to_dict(orient=\'records\')\n    if any(record.get(\'model_id\') != MODEL_ID for record in previous):\n        raise ValueError(\'CSV chứa mô hình khác. Chọn OUTPUT_DIR riêng.\')\n    saved_rows = {sample_key(record): record for record in previous}\nelse:\n    saved_rows = {}\nfinished = {key for key, record in saved_rows.items() if record.get(\'status\') == \'ok\'}\npending = [sample for sample in pairs if sample_key(sample) not in finished]\nprint(f\'Reusing {len(finished & selected_keys):,} successful selected pages; {len(pending):,} remaining.\', flush=True)\nevaluation_started = time.perf_counter()\nlast_progress_log = evaluation_started\nsuccessful_count = len(finished & selected_keys)\nerrors_this_run = 0\n\n\ndef format_duration(seconds):\n    hours, remainder = divmod(int(seconds), 3600)\n    minutes, seconds = divmod(remainder, 60)\n    return f\'{hours:02d}:{minutes:02d}:{seconds:02d}\'\n\n\nfor attempted_this_run, sample in enumerate(tqdm(pending, desc=\'Tesseract English fast evaluation\', unit=\'page\', mininterval=60), start=1):\n    if MAX_EVAL_HOURS is not None and time.perf_counter() - evaluation_started >= MAX_EVAL_HOURS * 3600:\n        print(\'Time budget reached. Export this checkpoint, then resume later.\', flush=True)\n        break\n    result = dict(sample)\n    result[\'model_id\'] = MODEL_ID\n    try:\n        ground_truth = \' \'.join(read_gt_tokens(sample[\'gt_path\']))\n        if not clean_text(ground_truth):\n            raise ValueError(\'GT chỉ chứa nhãn cấu trúc; không có văn bản để chấm điểm.\')\n        prediction, elapsed, n_regions = transcribe_page(sample[\'image_path\'])\n        result.update({\n            \'status\': \'ok\',\n            \'prediction\': prediction,\n            \'ground_truth\': ground_truth,\n            \'latency_sec\': elapsed,\n            \'n_text_regions\': n_regions,\n            **compute_metrics(ground_truth, prediction),\n        })\n    except Exception as exc:\n        result.update({\'status\': \'error\', \'error\': repr(exc)})\n        print(\'Failure:\', sample[\'sample_id\'], repr(exc), flush=True)\n        gc.collect()\n    saved_rows[sample_key(sample)] = result\n    temporary_csv = RESULTS_CSV.with_suffix(\'.csv.tmp\')\n    pd.DataFrame(saved_rows.values()).to_csv(temporary_csv, index=False)\n    temporary_csv.replace(RESULTS_CSV)\n    successful_count += result[\'status\'] == \'ok\'\n    errors_this_run += result[\'status\'] == \'error\'\n    now = time.perf_counter()\n    if attempted_this_run == 1 or now - last_progress_log >= 60 or attempted_this_run == len(pending):\n        elapsed = now - evaluation_started\n        average_sec = elapsed / attempted_this_run\n        unattempted = len(pending) - attempted_this_run\n        print(\n            f\'PROGRESS: {successful_count:,}/{len(pairs):,} successful ({100 * successful_count / len(pairs):.1f}%) | \'\n            f\'attempted this run={attempted_this_run:,}/{len(pending):,} | errors this run={errors_this_run:,} | \'\n            f\'elapsed={format_duration(elapsed)} | avg={average_sec:.2f}s/page | \'\n            f\'ETA for unattempted pages={format_duration(unattempted * average_sec)}\',\n            flush=True,\n        )\n        last_progress_log = now\n\nrows = [record for key, record in saved_rows.items() if key in selected_keys]\nresults = pd.DataFrame(rows)\nif results.empty:\n    raise RuntimeError(\'Chưa có ảnh được thử. Kiểm tra giới hạn thời gian và checkpoint.\')\nprint(\'Saved:\', RESULTS_CSV, flush=True)\nprint(\'Successful:\', (results.status == \'ok\').sum(), \'/\', len(pairs), \'selected pages\', flush=True)\ndisplay(results.reindex(columns=[\n    \'category\', \'occlusion_type\', \'cer\', \'wer\', \'token_f1\',\n    \'latency_sec\', \'n_text_regions\', \'status\', \'error\'\n]).head(10))\n\nprint(\'Summaries\', flush=True)\nok = results.loc[results[\'status\'] == \'ok\'].copy()\nassert len(ok), \'Không có trang chấm điểm được. Kiểm tra cột error.\'\nmetric_cols = [\'cer\', \'wer\', \'token_precision\', \'token_recall\', \'token_f1\', \'latency_sec\']\n# CSV có hàng lỗi có thể khiến pandas đọc các số thành chuỗi khi resume.\nfor column in metric_cols + [\'n_text_regions\']:\n    ok[column] = pd.to_numeric(ok[column], errors=\'raise\')\n\n\ndef aggregate_metrics(df, group_by):\n    return df.groupby(group_by, dropna=False).agg(\n        n=(\'sample_id\', \'count\'),\n        **{f\'mean_{name}\': (name, \'mean\') for name in metric_cols},\n        mean_text_regions=(\'n_text_regions\', \'mean\'),\n    ).reset_index()\n\n\noverall = {f\'mean_{name}\': float(ok[name].mean()) for name in metric_cols}\noverall.update({\n    \'n_selected\': int(len(pairs)),\n    \'n_success\': int(len(ok)),\n    \'n_errors\': int(len(results) - len(ok)),\n    \'n_remaining\': int(len(pairs) - len(ok)),\n    \'evaluation_complete\': bool(len(ok) == len(pairs)),\n    \'n_empty_predictions\': int(ok[\'prediction\'].fillna(\'\').astype(str).str.strip().eq(\'\').sum()),\n    \'mean_text_regions\': float(ok[\'n_text_regions\'].mean()),\n})\nby_category = aggregate_metrics(ok, [\'category\'])\nby_type = aggregate_metrics(ok, [\'category\', \'occlusion_type\'])\nby_density = aggregate_metrics(ok, [\'category\', \'occlusion_density\'])\nprint(\'OVERALL macro-averaged metrics:\', flush=True)\nprint(json.dumps(overall, indent=2), flush=True)\nprint(\'BY OCCLUSION REGION:\', flush=True)\ndisplay(by_category)\nprint(\'BY OCCLUSION TYPE:\', flush=True)\ndisplay(by_type)\nprint(\'BY OCCLUSION DENSITY:\', flush=True)\ndisplay(by_density)\nby_category.to_csv(OUTPUT_DIR / \'summary_by_category.csv\', index=False)\nby_type.to_csv(OUTPUT_DIR / \'summary_by_occlusion_type.csv\', index=False)\nby_density.to_csv(OUTPUT_DIR / \'summary_by_density.csv\', index=False)\n\nregion_times = by_category.set_index(\'category\')[\'mean_latency_sec\']\nif {\'text\', \'nontext\'}.issubset(region_times.index):\n    estimated_full_hours = (6000 * region_times[\'text\'] + 3000 * region_times[\'nontext\']) / 3600\n    print(f\'Ước lượng suy luận 9.000 trang trên {DEVICE_NAME}: {estimated_full_hours:.1f} giờ.\', flush=True)\nprint(\'Số ảnh còn cần chấm thành công:\', overall[\'n_remaining\'], flush=True)\n\n\n# Điểm theo chính sách cũ để đối chiếu các CSV trước khi lọc marker.\nraw_metric_cols = [\'cer\', \'wer\', \'token_precision\', \'token_recall\', \'token_f1\']\nfor name in raw_metric_cols + [\'gt_words\', \'pred_words\']:\n    ok[f\'raw_{name}\'] = pd.to_numeric(ok[f\'raw_{name}\'], errors=\'raise\')\nfor column in [\'n_gt_markers_removed\', \'n_pred_markers_removed\']:\n    ok[column] = pd.to_numeric(ok[column], errors=\'raise\')\nraw_overall = {f\'mean_{name}\': float(ok[f\'raw_{name}\'].mean()) for name in raw_metric_cols}\nraw_overall[\'mean_latency_sec\'] = overall[\'mean_latency_sec\']\noverall[\'n_gt_markers_removed\'] = int(ok[\'n_gt_markers_removed\'].sum())\noverall[\'n_pred_markers_removed\'] = int(ok[\'n_pred_markers_removed\'].sum())\nraw_ok = ok.copy()\nfor name in raw_metric_cols:\n    raw_ok[name] = raw_ok[f\'raw_{name}\']\nfor filename, groups in [\n    (\'raw_summary_by_category.csv\', [\'category\']),\n    (\'raw_summary_by_occlusion_type.csv\', [\'category\', \'occlusion_type\']),\n    (\'raw_summary_by_density.csv\', [\'category\', \'occlusion_density\']),\n]:\n    aggregate_metrics(raw_ok, groups).to_csv(OUTPUT_DIR / filename, index=False)\nprint(\'RAW/LEGACY macro metrics (còn nhãn, để đối chiếu CSV cũ):\', flush=True)\nprint(json.dumps(raw_overall, indent=2), flush=True)\nprint(\'GT marker tokens removed:\', overall[\'n_gt_markers_removed\'], flush=True)\nprint(\'Không so trực tiếp điểm chính đã lọc với điểm gốc chưa lọc của model khác.\', flush=True)\n\nprint(\'Save plots\', flush=True)\nax = by_category.set_index(\'category\')[[\'mean_cer\', \'mean_wer\']].plot(\n    kind=\'bar\', figsize=(7, 4), rot=0, title=\'OPRB test: mean CER and WER by region\'\n)\nax.set_xlabel(\'Occlusion region\')\nax.set_ylabel(\'Error rate (lower is better)\')\nplt.tight_layout()\nplt.savefig(OUTPUT_DIR / \'cer_wer_by_category.png\', dpi=150)\nplt.close()\n\nax = by_type.set_index(\'occlusion_type\').groupby(level=0)[\'mean_wer\'].mean().sort_values().plot(\n    kind=\'barh\', figsize=(9, 5), title=\'Mean WER by occlusion type (unweighted category average)\'\n)\nax.set_xlabel(\'Mean word error rate (lower is better)\')\nplt.tight_layout()\nplt.savefig(OUTPUT_DIR / \'wer_by_occlusion_type.png\', dpi=150)\nplt.close()\n\nprint(\'Export results\', flush=True)\nrun_config = {\n    **checkpoint_settings,\n    \'categories\': CATEGORIES,\n    \'no_finetuning\': True,\n    \'max_samples_per_category\': MAX_SAMPLES_PER_CATEGORY,\n    \'random_seed\': RANDOM_SEED,\n    \'max_eval_hours_per_run\': MAX_EVAL_HOURS,\n    \'metrics\': overall,\n    \'raw_metrics\': raw_overall,\n    \'ground_truth_storage\': \'Original GT tokens retained in ground_truth; layout tokens removed only while scoring\',\n    \'gpu_name\': None,\n    \'cpu_info\': CPU_INFO,\n    \'cuda_version\': None,\n    \'python_version\': platform.python_version(),\n    \'platform\': platform.platform(),\n    \'runtime_python\': sys.executable,\n    \'torch_available_in_runtime\': False,\n    \'runtime_packages_file\': \'runtime_packages.json\',\n    \'jiwer_version\': version(\'jiwer\'),\n    \'pandas_version\': version(\'pandas\'),\n    \'numpy_version\': np.__version__,\n    \'output_token_limit\': None,\n    \'output_token_limit_note\': \'No VLM generation-token cap; page segmentation, vocabulary and per-page recognition timeout still apply.\',\n}\n(OUTPUT_DIR / \'run_config.json\').write_text(json.dumps(run_config, indent=2), encoding=\'utf-8\')\nzip_name = shutil.make_archive(str(OUTPUT_DIR), \'zip\', root_dir=OUTPUT_DIR)\nprint(\'Results ZIP:\', zip_name, flush=True)\nprint(\'Tải ZIP trong Output của Kaggle.\', flush=True)\nprint(\'Lần sau: gắn output phiên này làm Input rồi đặt RESUME_FROM đến thư mục CSV/JSON.\', flush=True)\nif not overall[\'evaluation_complete\']:\n    print(\'Đây là kết quả một phần. Tiếp tục checkpoint để hoàn tất tập đã chọn.\', flush=True)\n'
BENCHMARK_SCRIPT = OUTPUT_DIR / 'isolated_benchmark.py'
BENCHMARK_SCRIPT.write_text(BENCHMARK_SOURCE, encoding='utf-8')
CONFIG_JSON = OUTPUT_DIR / 'benchmark_settings.json'
CONFIG_JSON.write_text(json.dumps(RUN_SETTINGS, indent=2), encoding='utf-8')
print('Benchmark script:', BENCHMARK_SCRIPT, flush=True)
print('SHA-256:', hashlib.sha256(BENCHMARK_SOURCE.encode()).hexdigest(), flush=True)

In [ ]:
import os
import signal
import subprocess

child_env = os.environ.copy()
for key in ('PYTHONPATH', 'PYTHONHOME', 'LD_PRELOAD'):
    child_env.pop(key, None)
child_env['PYTHONNOUSERSITE'] = '1'
# Ưu tiên thư viện native Tesseract đi kèm wheel, không dùng CUDA.
library_dirs = sorted(TESSERACT_ENV.glob('lib/python*/site-packages/tesserocr.libs'))
inherited_paths = [path for path in child_env.get('LD_LIBRARY_PATH', '').split(os.pathsep)
                   if path and 'site-packages' not in path and 'dist-packages' not in path]
child_env['LD_LIBRARY_PATH'] = os.pathsep.join([str(path) for path in library_dirs] + inherited_paths)
LOG_FILE = OUTPUT_DIR / 'execution.log'
command = [str(TESSERACT_PYTHON), '-I', '-u', str(BENCHMARK_SCRIPT), '--config', str(CONFIG_JSON)]
print('Bắt đầu benchmark. Theo dõi PROGRESS bên dưới hoặc View Logs của Saved Version.', flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1, env=child_env, start_new_session=True)
try:
    with LOG_FILE.open('a', encoding='utf-8') as log:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
    returncode = process.wait()
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)
    print('Đã dừng. Checkpoint của các trang ghi xong vẫn còn; chạy lại Mục 3 để tiếp tục.', flush=True)
    raise
finally:
    if process.poll() is None:
        process.terminate()
    if process.stdout is not None:
        process.stdout.close()
if returncode:
    raise RuntimeError(f'Benchmark exited with code {returncode}. Xem traceback phía trên và {LOG_FILE}.')
print('Hoàn tất lượt chạy. Xem kết quả ở Mục 4.', flush=True)

## 4. Xem và tải kết quả

In [ ]:
import csv
import html
import json
from IPython.display import display, HTML, Image, FileLink

config_path = OUTPUT_DIR / 'run_config.json'
if config_path.is_file():
    config = json.loads(config_path.read_text(encoding='utf-8'))
    print('MAIN METRICS (đã lọc nhãn GT):', json.dumps(config['metrics'], indent=2), flush=True)
    print('RAW METRICS (đối chiếu CSV cũ):', json.dumps(config['raw_metrics'], indent=2), flush=True)
    for filename in ['summary_by_category.csv', 'summary_by_occlusion_type.csv']:
        with (OUTPUT_DIR / filename).open(encoding='utf-8', newline='') as stream:
            rows = list(csv.reader(stream))
        table = '<table>' + ''.join('<tr>' + ''.join('<td>' + html.escape(value) + '</td>' for value in row) + '</tr>' for row in rows) + '</table>'
        print(filename, flush=True)
        display(HTML(table))
    for filename in ['cer_wer_by_category.png', 'wer_by_occlusion_type.png', 'preview.png']:
        path = OUTPUT_DIR / filename
        if path.is_file(): display(Image(filename=str(path)))
    archive = OUTPUT_DIR.with_suffix('.zip')
    if archive.is_file(): display(FileLink(str(archive)))
    print('Tải ZIP/thư mục trong Output của Kaggle. Saved Version mới cần RESUME_FROM.', flush=True)
else:
    print('Chưa có run_config.json. Kiểm tra log và checkpoint per_image_results.csv.', flush=True)

## Ghi báo cáo

- Tesseract English LSTM là một model pretrained thứ năm; không tính riêng page segmentation thành model thứ sáu. Bộ `tessdata_fast` dùng mạng LSTM dạng integer; không phải tự lượng tử hóa hoặc huấn luyện lại.
- Ghi engine Tesseract/Leptonica, tesserocr, CPU, giới hạn thread, OEM 1, PSM 3, DPI metadata và hash/revision trọng số. `run_config.json`, `model_manifest.json`, `runtime_packages.json` lưu thông tin tái hiện. `traineddata` chứa cả mạng/từ điển; báo kích thước MiB, không suy số tham số bằng dung lượng chia 4.
- Model chỉ nhận ảnh che khuất. Không resize ảnh, dùng GT để chọn vùng hoặc sửa transcript. Tắt học thích nghi và từ điển tài liệu; reset trạng thái giữa các trang.
- Giữ GT/prediction gốc. Chỉ số chính bỏ đúng hai token cấu trúc nguyên vẹn `##LTLine##`, `##LTFigure##`, như PaddleOCR. Các cột/bảng `raw_*` còn nhãn để đối chiếu các CSV cũ. Chấm lại transcript của Florence/EasyOCR/docTR bằng cùng chính sách nếu so sánh điểm chính; không cần suy luận lại.
- CER/WER/P/R/F1 là tỷ lệ, nhân 100 khi trình bày %. Tổng hợp macro theo ảnh; token F1 là giao đa tập, không phải F1 hộp chữ. `n_text_regions` ở đây đếm số word confidences Tesseract trả về; không lọc từ bằng confidence.
- Dự đoán rỗng vẫn được chấm điểm. Timeout/lỗi được lưu riêng và thử lại khi resume; không coi lỗi thực thi là một dự đoán rỗng.
- 10 ảnh chỉ là chạy thử. `evaluation_complete=True` nói về tập đang chọn; kiểm tra `n_selected` trước khi ghi hoàn tất 9.000 ảnh. Không suy tốc độ toàn bộ OPRB từ ảnh kiểm tra đơn giản; dùng kết quả thử trên cùng 10 ảnh để ngoại suy và ghi rõ giới hạn.
- Báo cáo dưới bốn trang A4, giữ template hiện có. Cần bảng chỉ số, so sánh text/nontext, loại/mật độ che, ví dụ đúng/sai và đánh đổi chất lượng/kích thước/thời gian.

Nguồn: [Tesseract API](https://tesseract-ocr.github.io/tessdoc/APIExample.html), [tessdata_fast](https://github.com/tesseract-ocr/tessdata_fast), [tesserocr 2.13.0](https://pypi.org/project/tesserocr/2.13.0/).